# Poster evidence notebook (unexecuted)

**Status:** saved with empty outputs and null execution counts. Nothing in this notebook has been executed or verified by Claude Code.

**Purpose:** inspect the evidence behind every poster number, figure and claim, using only files your own runs generated.

**Inputs (generated by you, see `POSTER_RUNBOOK.md`):**
* `reports/poster/pilot_audit.json` - step 7 (`RECOMPUTED_PILOT`)
* `reports/poster/smoke_validation.json` and the derived files it points to - steps 9-10 (`NEW_POSTER_SMOKE`)
* `reports/poster/rewrite_verification_summary.json` - step 11 (`NEW_POSTER_SMOKE`, `MANUAL_REFERENCE_REWRITE`)
* `reports/poster/environment/*.json` - steps 4-6b (provenance)

**Expected outputs:** tables and figures shown inline only. Files for the poster are written by `scripts/powershell/06_build_poster_evidence.ps1`, not by this notebook.

**Rules:**
* Run the cells one at a time, top to bottom.
* A cell without its evidence raises `MissingEvidenceError` naming the runbook step to run. There is no fallback or demonstration data.
* All logic lives in the `query_cost_predictor` package; cells only call it.
* Stop at every **Checkpoint** and inspect the counts before continuing.

In [ ]:
# 0.1 Setup: load .env (for QCP_DATA_ROOT) and whatever evidence exists. Nothing is generated here.
%matplotlib inline
import pandas as pd
from IPython.display import display

from query_cost_predictor.envfile import load_repo_env
from query_cost_predictor.errors import MissingEvidenceError
from query_cost_predictor.reporting import analysis as A
from query_cost_predictor.reporting import builder as B
from query_cost_predictor.reporting import figures as F
from query_cost_predictor.reporting import loaders as L
from query_cost_predictor.reporting import pilot_figures as PF

load_repo_env()
bundle = L.load_all()

In [ ]:
# 0.2 Which evidence sources are present? Missing sources name the step that produces them.
display(pd.DataFrame([
    {"source": "pilot audit (RECOMPUTED_PILOT)", "present": bundle.pilot is not None, "problem": bundle.problems.get("pilot", "")},
    {"source": "poster smoke (NEW_POSTER_SMOKE)", "present": bundle.smoke is not None, "problem": bundle.problems.get("smoke", "")},
    {"source": "manual rewrites (NEW_POSTER_SMOKE)", "present": bundle.rewrite is not None, "problem": bundle.problems.get("rewrite", "")},
]))

## 1. Evidence provenance and checksums

Every evidence file is re-hashed and compared with the SHA-256 recorded when it was produced (pilot inputs, derived smoke files, raw JSONL, rewrite raw file).

In [ ]:
# 1.1 Recompute SHA-256 values; a False in `matches` means a file changed after it was recorded.
provenance = pd.DataFrame(A.provenance_table(bundle))
display(provenance)

In [ ]:
# 1.2 Stop if any recorded checksum no longer matches.
changed = provenance[provenance["matches"] == False]  # noqa: E712 - compare with False explicitly (None = not recorded)
if len(changed):
    raise RuntimeError(f"{len(changed)} evidence file(s) changed after they were recorded - investigate before continuing")
print("All recorded checksums match.")

**Checkpoint 1:** every `matches` value is `True` or empty. Do not continue otherwise.

## 2. Dataset integrity

In [ ]:
# 2.1 Pilot data-quality checks recomputed by the audit (explain any FAIL before using a pilot number).
pilot = L.require(bundle.pilot, bundle.problems.get("pilot", "pilot audit missing"))
display(pd.DataFrame(pilot.audit["data_quality"]))

In [ ]:
# 2.2 Poster-smoke validation checks (FAIL blocks use; OBSERVED_ABSENT is an honest observation, not a failure).
smoke = L.require(bundle.smoke, bundle.problems.get("smoke", "smoke evidence missing"))
display(pd.DataFrame(smoke.validation["checks"]))

**Checkpoint 2:** no `FAIL` in either table. Read every `WARN` and `OBSERVED_ABSENT` row.

## 3. Query, group and repetition counts

A modeling key is one query instance on one snapshot and configuration under one protocol. Repeated executions are child records and are never counted as extra keys.

In [ ]:
# 3.1 Pilot counts (instances, templates, raw runs, distinct SQL, structural fingerprints, plan shapes).
display(pd.DataFrame(A.pilot_counts(L.require(bundle.pilot, bundle.problems.get("pilot", "")).audit)))

In [ ]:
# 3.2 Poster-smoke counts (keys, labelled keys, groups, templates, measured executions).
display(pd.DataFrame(A.smoke_counts(L.require(bundle.smoke, bundle.problems.get("smoke", "")))))

**Checkpoint 3:** the smoke run has 20-50 keys; measured executions are about three per labelled key and are not reported as keys.

## 4. Runtime distribution

In [ ]:
# 4.1 Log-scale runtime distribution; pilot and smoke are separate panels and are never pooled.
runtime_panels = []
if bundle.pilot is not None:
    runtime_panels.append(PF.runtime_panel(bundle.pilot.audit))
if bundle.smoke is not None:
    runtime_panels.append(B.smoke_runtime_panel(bundle.smoke))
display(F.runtime_distribution_figure(runtime_panels))  # raises MissingEvidenceError when both are absent

In [ ]:
# 4.2 Pilot runtime summary (median of three runs per instance, ms).
recomputed = L.require(bundle.pilot, bundle.problems.get("pilot", "")).audit["recomputed"]
display(pd.Series({k.split(".")[-1]: v for k, v in recomputed.items() if k.startswith("runtime.median_label.")}))

In [ ]:
# 4.3 Smoke runtime-bin coverage (complete keys by bin; censored and failed keys listed separately).
display(pd.DataFrame(A.smoke_bin_table(L.require(bundle.smoke, bundle.problems.get("smoke", "")))))

## 5. PostgreSQL estimated cost versus measured runtime

Estimated cost is in planner units, not milliseconds or money; only rank agreement is meaningful without calibration.

In [ ]:
# 5.1 Cost versus runtime (log-log); censored smoke keys are drawn at their timeout lower bound.
scatter_panels = []
if bundle.pilot is not None:
    scatter_panels.append(PF.scatter_panel(bundle.pilot.audit))
if bundle.smoke is not None:
    scatter_panels.append(B.smoke_scatter_panel(bundle.smoke))
display(F.cost_vs_runtime_figure(scatter_panels))

In [ ]:
# 5.2 Pilot rank correlations between raw estimated cost and runtime (no fitting involved).
display(pd.Series(L.require(bundle.pilot, bundle.problems.get("pilot", "")).audit["models"]["raw_postgres_cost"]))

## 6. Random versus grouped evaluation (pilot only)

Template-grouped 5-fold is the primary estimate. The random split is shown only as a leakage diagnostic. All models share identical folds; template ids are never features.

In [ ]:
# 6.1 Pooled out-of-fold metrics with keys, groups and keys per group (R2 is secondary, never 'accuracy').
metrics = pd.DataFrame(PF.metric_rows(L.require(bundle.pilot, bundle.problems.get("pilot", "")).audit))
columns = ["model", "split", "n_keys", "n_groups", "keys_per_group", "median_qerror", "p90_qerror", "p95_qerror",
           "log_mae", "mae_ms", "rmse_ms", "r2_log", "r2_ms"]
display(metrics[metrics["scope"] == "pooled_out_of_fold"][columns])

In [ ]:
# 6.2 Random versus grouped chart (identical folds for all models).
audit = L.require(bundle.pilot, bundle.problems.get("pilot", "pilot audit missing")).audit
display(F.random_vs_grouped_figure(PF.metric_rows(audit), PF.pilot_provenance(audit, "pooled out-of-fold median q-error and log MAE")))

In [ ]:
# 6.3 Re-check that no template appears in more than one grouped test fold, and show fold sizes.
from query_cost_predictor.splits import assert_groups_disjoint
assignment = audit["models"]["folds"]["grouped"]["assignment"]
template_of = {i["query_id"]: i["template_id"] for i in audit["instances"]}
assert_groups_disjoint([assignment[q] for q in assignment], [template_of[q] for q in assignment])
display(pd.DataFrame(audit["models"]["folds"]["grouped"]["summary"]))

**Checkpoint 6:** grouped folds are disjoint by template; the poster quotes grouped metrics with keys and groups next to them.

## 7. Runtime-bin and tail-error analysis

In [ ]:
# 7.1 Pilot out-of-fold q-error by actual-runtime bin under template-grouped folds (primary).
display(pd.DataFrame(A.pilot_tail_table(L.require(bundle.pilot, bundle.problems.get("pilot", "")).audit, "grouped")))

In [ ]:
# 7.2 The same table under random folds (leakage diagnostic only).
display(pd.DataFrame(A.pilot_tail_table(L.require(bundle.pilot, bundle.problems.get("pilot", "")).audit, "random")))

## 8. Operator frequencies (plain, estimated plans)

In [ ]:
# 8.1 Share of keys whose estimated plan contains each node type.
display(pd.DataFrame(A.operator_table(bundle)))

In [ ]:
# 8.2 Operator coverage chart (one panel per evidence source).
operator_panels = []
if bundle.pilot is not None:
    operator_panels.append(PF.operator_panel(bundle.pilot.audit))
if bundle.smoke is not None:
    operator_panels.append(B.smoke_operator_panel(bundle.smoke))
display(F.operator_frequency_figure(operator_panels))

## 9. Spill, disk-read and parallel-worker evidence (execution-only, class E: diagnostics, never features)

In [ ]:
# 9.1 Pilot resource evidence recomputed from raw plans.
pilot_recomputed = L.require(bundle.pilot, bundle.problems.get("pilot", "")).audit["recomputed"]
display(pd.Series({k: v for k, v in pilot_recomputed.items() if k.startswith("resources.")}))

In [ ]:
# 9.2 Smoke resource evidence per key (measured runs only).
display(pd.DataFrame(A.smoke_resource_table(L.require(bundle.smoke, bundle.problems.get("smoke", "")))))

## 10. High-runtime class feasibility

No classifier is trained here. With zero (or too few) positive examples a high-runtime classifier is `NOT_YET_EVALUABLE` and must not appear on the poster.

In [ ]:
# 10.1 Positives per threshold and evidence source (censored smoke keys count as exceeding the timeout).
display(pd.DataFrame(A.high_runtime_feasibility(bundle)))

## 11. Manual reference-rewrite verification results

Label: `MANUAL_REFERENCE_REWRITE` (hand-written; not LLM output). Equal results on a snapshot are evidence, not proof, of equivalence.

In [ ]:
# 11.1 Every check, including the negative control that must be rejected.
L.require(bundle.rewrite, bundle.problems.get("rewrite", "rewrite verification missing"))
display(pd.DataFrame(B.rewrite_rows(bundle)))

In [ ]:
# 11.2 Paired speedups, only for reference pairs that passed equivalence with complete timings.
timings, rewrite_provenance = B.rewrite_speedup_inputs(bundle)  # raises MissingEvidenceError otherwise
display(F.rewrite_speedup_figure(timings, rewrite_provenance))

**Checkpoint 11:** the negative control is `NOT_EQUIVALENT` / `DO_NOT_RECOMMEND`. If it is not, the verification method failed; stop.

## 12. Poster-safe claim table

Claims render only from evidence. `EVIDENCE_MISSING` claims must not be used; `PROHIBITED_WORDING` means a rule in `docs/poster_claim_register.md` was violated.

In [ ]:
# 12.1 Render all claims from the loaded evidence and show their status.
claims_df = pd.DataFrame(A.claim_table(bundle))
display(claims_df[["id", "status", "render_status", "text", "missing", "violations"]])

In [ ]:
# 12.2 Lint any caption you intend to write by hand before putting it on the poster.
caption = ""  # paste your caption text here
print(B.lint_free_text(caption) if caption else "No caption entered.")

## 13. Known limitations and next steps

* Pilot: SF 0.1, one machine, one session, warm cache; plain `EXPLAIN` was not captured, so D-class features come from the estimate view of `EXPLAIN ANALYZE` plans.
* Smoke: 20-50 keys demonstrate the protocol (estimate/execution separation, censoring, resumability); they do not measure generalization.
* Instrumented `EXPLAIN ANALYZE` server time is not end-to-end latency; the client-latency comparison is planned for calibration.
* RQ2 (LLM rewrites) is not yet evaluable; only manual reference rewrites were verified.
* Next steps (planned): calibration, the approximately 4,000-5,000-key final collection across at least 150 groups, grouped evaluation, and the gated rewrite study.

In [ ]:
# 13.1 Evidence still missing (each entry names the runbook step that produces it).
display(pd.Series(bundle.problems, dtype="object") if bundle.problems else "All three evidence sources are present.")